In [1]:
from pathlib import Path
import sqlite3
import polars as pl

In [2]:
db_path = Path("ecommerce.sqlite")
staging = Path("spark_input")
staging.mkdir(exist_ok=True)

TABLES = [
    "orders",
    "order_items",
    "products",
    "customers",
    "order_payments",
    "order_reviews",
    "product_category_name_translation",
]

with sqlite3.connect(db_path) as conn:
    for table in TABLES:
        df = pl.read_database(
            f'SELECT * FROM "{table}"',
            connection=conn
        )
        df.write_csv(
            staging / (table + ".csv"),
        )
        print(f"{table:36s} {len(df):>8} rows")

orders                                  99441 rows
order_items                            112650 rows
products                                32951 rows
customers                               99441 rows
order_payments                         103886 rows
order_reviews                           97621 rows
product_category_name_translation          71 rows


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Practice4_Ecommerce")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/19 16:59:13 WARN Utils: Your hostname, alex-VivoBook-ASUSLaptop-M1603QA-M1603QA, resolves to a loopback address: 127.0.1.1; using 10.119.22.238 instead (on interface wlp1s0)
26/09/19 16:59:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/alex/git/MIREA/InfAndCompTechnology/Bachelor/4 course/1 Term/DE/3/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/19 16:59:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-j

Master: local[*]
App: Practice4_Ecommerce
Spark: 4.2.0


In [4]:
orders_raw = spark.read.option("header", True).option("encoding", "UTF-8").csv(str(staging / "orders.csv"))

orders_raw.printSchema()
print(f"orders_raw: {orders_raw.count():,} rows")

orders_raw.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
).show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

orders_raw: 99,441 rows
+--------------------------------+--------------------------------+------------+------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|
+--------------------------------+--------------------------------+------------+------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |
|47770eb9100c2d0c44946d9cf

In [5]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])
items = (
    spark.read
    .option("header", True)
    .schema(order_items_schema)
    .csv(str(staging / "order_items.csv"))
)
items.printSchema()
print("Rows:", items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Rows: 112650


In [6]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = orders_raw
for c in timestamp_cols:
    orders = orders.withColumn(
        c,
        F.to_timestamp(F.col(c), "yyyy-MM-dd HH:mm:ss")
    )
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [7]:
expensive_items = (
    items
    .filter(F.col("price") >= 500)
    .withColumn(
        "gross_item_value",
        F.round(F.col("price") + F.col("freight_value"), 2)
    )
    .select(
        "order_id", "order_item_id",
        "price", "freight_value", "gross_item_value"
    )
)
print("Items with price >= 500:", expensive_items.count())
expensive_items.orderBy(
    F.desc("gross_item_value")
).show(5, truncate=False)

Items with price >= 500: 3239
+--------------------------------+-------------+------+-------------+----------------+
|order_id                        |order_item_id|price |freight_value|gross_item_value|
+--------------------------------+-------------+------+-------------+----------------+
|0812eb902a67711a1cb742b3cdaa65ae|1            |6735.0|194.31       |6929.31         |
|fefacc66af859508bf1a7934eab1e97f|1            |6729.0|193.21       |6922.21         |
|f5136e38d1a14a4dbd87dff67da82701|1            |6499.0|227.66       |6726.66         |
|a96610ab360d42a2e5335a3998b4718a|1            |4799.0|151.34       |4950.34         |
|199af31afc78c699f0dbf71fb178d4d4|1            |4690.0|74.34        |4764.34         |
+--------------------------------+-------------+------+-------------+----------------+
only showing top 5 rows


In [8]:
delivered = (
    orders
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "is_late",
        F.col("order_delivered_customer_date") >
        F.col("order_estimated_delivery_date")
    )
)
with_delivery_date = delivered.filter(
    F.col("order_delivered_customer_date").isNotNull()
)
late = with_delivery_date.filter(F.col("is_late"))

print("Доставленных заказов:", delivered.count())
print("С известной датой доставки:", with_delivery_date.count())
print("Позже ожидаемой даты:", late.count())

Доставленных заказов: 96478
С известной датой доставки: 96470
Позже ожидаемой даты: 7826


In [9]:
null_counts = orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in orders.columns
])

null_counts.show(vertical=True)

-RECORD 0-----------------------------
 order_id                      | 0    
 customer_id                   | 0    
 order_status                  | 0    
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



In [10]:
missing_delivery_by_status = (
    orders
    .filter(F.col("order_delivered_customer_date").isNull())
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

missing_delivery_by_status.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |619  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|delivered   |8    |
|created     |5    |
|approved    |2    |
+------------+-----+



In [11]:
products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(staging / "products.csv"))
)
missing_categories = products.filter(
    F.col("product_category_name").isNull()
).count()
products_clean = products.fillna({
    "product_category_name": "unknown"
})
print("Товаров без категории:", missing_categories)

Товаров без категории: 610


In [12]:
status_stats = (
    orders
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

status_stats.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



In [13]:
payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(staging / "order_payments.csv"))
)
payment_stats = (
    payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("payments"),
        F.round(F.sum("payment_value"), 2).alias("total_value"),
        F.round(F.avg("payment_value"), 2).alias("avg_value")
    )
    .orderBy(F.desc("total_value"))
)
payment_stats.show(truncate=False)

+------------+--------+-------------+---------+
|payment_type|payments|total_value  |avg_value|
+------------+--------+-------------+---------+
|credit_card |76795   |1.254208419E7|163.32   |
|boleto      |19784   |2869361.27   |145.03   |
|voucher     |5775    |379436.87    |65.7     |
|debit_card  |1529    |217989.79    |142.57   |
|not_defined |3       |0.0          |0.0      |
+------------+--------+-------------+---------+



In [14]:
translations = (
    spark.read
    .option("header", True)
    .csv(str(staging / "product_category_name_translation.csv"))
)
sales = (
    items.alias("i")
    .join(
        products_clean.select(
            "product_id", "product_category_name"
        ).alias("p"),
        F.col("i.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(
        translations.alias("t"),
        F.col("p.product_category_name") == F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.col("i.order_id").alias("order_id"),
        F.col("i.order_item_id").alias("order_item_id"),
        F.col("i.product_id").alias("product_id"),
        F.col("i.price").alias("price"),
        F.col("i.freight_value").alias("freight_value"),
        F.coalesce(
            F.col("t.product_category_name_english"),
            F.col("p.product_category_name"),
            F.lit("unknown")
        ).alias("category")
    )
)

print("Строк до JOIN:", items.count())
print("Строк после JOIN:", sales.count())
print(
    "Позиций категории unknown:",
    sales.filter(F.col("category") == "unknown").count()
)

Строк до JOIN: 112650
Строк после JOIN: 112650
Позиций категории unknown: 1603


In [15]:
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(staging / "customers.csv"))
)

late_by_state = (
    delivered
    .join(
        customers.select("customer_id", "customer_state"),
        on="customer_id",
        how="left"
    )
    .filter(F.col("order_delivered_customer_date").isNotNull())
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("delivered"),
        F.sum(
        F.when(F.col("is_late"), 1).otherwise(0)
        ).alias("late")
    )
    .withColumn(
        "late_pct",
        F.round(F.col("late") / F.col("delivered") * 100, 2)
    )
    .orderBy(F.desc("late_pct"))
)

late_by_state.show(10, truncate=False)

+--------------+---------+----+--------+
|customer_state|delivered|late|late_pct|
+--------------+---------+----+--------+
|AL            |397      |95  |23.93   |
|MA            |717      |141 |19.67   |
|PI            |476      |76  |15.97   |
|CE            |1279     |196 |15.32   |
|SE            |335      |51  |15.22   |
|BA            |3256     |457 |14.04   |
|RJ            |12350    |1664|13.47   |
|TO            |274      |35  |12.77   |
|PA            |946      |117 |12.37   |
|ES            |1995     |244 |12.23   |
+--------------+---------+----+--------+
only showing top 10 rows


In [16]:
sales.createOrReplaceTempView("sales")

category_sql = spark.sql("""
    SELECT
    category,
    COUNT(*) AS items,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(price), 2) AS revenue,
    ROUND(AVG(price), 2) AS avg_price
    FROM sales
    GROUP BY category
    ORDER BY revenue DESC
    LIMIT 10
""")

category_sql.show(truncate=False)

+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+



In [17]:
category_sales = (
    sales
    .groupBy("category")
    .agg(
        F.count("*").alias("items"),
        F.countDistinct("order_id").alias("orders"),
        F.round(F.sum("price"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
)

print("Категорий в витрине:", category_sales.count())

category_sales.orderBy(
    F.desc("revenue")
).show(10, truncate=False)

Категорий в витрине: 74
+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+
only showing top 10 rows


In [18]:
OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

(
    category_sales
    .write
    .mode("overwrite")
    .parquet(str(OUT / "category_sales_parquet"))
)

(
    category_sales
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(OUT / "category_sales_csv"))
)

category_back = spark.read.parquet(
    str(OUT / "category_sales_parquet")
)

print("Строк после round-trip:", category_back.count())

category_back.printSchema()

Строк после round-trip: 74
root
 |-- category: string (nullable = true)
 |-- items: long (nullable = true)
 |-- orders: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- avg_price: double (nullable = true)



In [19]:
spark.stop()

# Датасет

Датасет о полетах в 2015 году, взят с [kaggle](https://www.kaggle.com/datasets/usdot/flight-delays?select=flights.csv)

In [20]:
spark = (
    SparkSession.builder
    .appName("Practice4_Flight")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)

data_path = Path("hw_input")


Master: local[*]
App: Practice4_Flight
Spark: 4.2.0


In [21]:
flights =(
    spark
    .read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(data_path / "flights.csv"))
).select(
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK",
    "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "DISTANCE",
    "SCHEDULED_DEPARTURE", "DEPARTURE_TIME",
    "SCHEDULED_ARRIVAL", "ARRIVAL_TIME",
    "CANCELLATION_REASON", "AIR_SYSTEM_DELAY", "SECURITY_DELAY",
    "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"
)

flights.show(5, truncate=False)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+--------+-------------------+--------------+-----------------+------------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DISTANCE|SCHEDULED_DEPARTURE|DEPARTURE_TIME|SCHEDULED_ARRIVAL|ARRIVAL_TIME|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+--------+-------------------+--------------+-----------------+------------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|2015|1    |1  |4          |AS     |98           |N407AS     |ANC           |SEA                |1448    |5                  |2354          |430              |408         |NULL     

In [22]:
airports = (
    spark
    .read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(data_path / "airports.csv"))
)

airports.show(5, truncate=False)

+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                            |CITY       |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|ABE      |Lehigh Valley International Airport|Allentown  |PA   |USA    |40.65236|-75.4404  |
|ABI      |Abilene Regional Airport           |Abilene    |TX   |USA    |32.41132|-99.6819  |
|ABQ      |Albuquerque International Sunport  |Albuquerque|NM   |USA    |35.04022|-106.60919|
|ABR      |Aberdeen Regional Airport          |Aberdeen   |SD   |USA    |45.44906|-98.42183 |
|ABY      |Southwest Georgia Regional Airport |Albany     |GA   |USA    |31.53552|-84.19447 |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
only showing top 5 rows


In [23]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [24]:
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField("IATA_CODE", T.StringType(), True),
    T.StructField("AIRPORT",   T.StringType(), True),
    T.StructField("CITY",      T.StringType(), True),
    T.StructField("STATE",     T.StringType(), True),
    T.StructField("COUNTRY",   T.StringType(), True),
    T.StructField("LATITUDE",  T.DoubleType(), True),
    T.StructField("LONGITUDE", T.DoubleType(), True),
])

airports = (spark.read
    .option("header", True)
    .option("mode", "FAILFAST")      # still worth it: catches unparseable doubles
    .schema(schema)
    .csv(str(data_path / "airports.csv"))
)

airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



Парсер корректно распознает все типы данных у airports, но у flights.csv некорректно данные обработал время. Однако Spark не поддерживает работу со временем, поэтому не будет исправлений

In [25]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- CANCELLATION_REASON: string (nullable = true)
 |-- AIR_SYSTEM_DELAY: integer (nullable = true)
 |-- SECURITY_DELAY: integer (nullable = true)
 |-- AIRLINE_DELAY: integer (nullable = true)
 |-- LATE_AIRCRAFT_DELAY: integer (nullable = true)
 |-- WEATHER_DELAY: integer (nullable = true)



In [26]:
airports.filter(F.col("STATE") == "CA").show(5, truncate=False)

+---------+----------------------------------------------+-------------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                                       |CITY         |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+----------------------------------------------+-------------+-----+-------+--------+----------+
|ACV      |Arcata Airport                                |Arcata/Eureka|CA   |USA    |40.97812|-124.10862|
|BFL      |Meadows Field                                 |Bakersfield  |CA   |USA    |35.4336 |-119.05677|
|BUR      |Bob Hope Airport (Hollywood Burbank Airport)  |Burbank      |CA   |USA    |34.20062|-118.3585 |
|CEC      |Del Norte County Airport (Jack McNamara Field)|Crescent City|CA   |USA    |41.78016|-124.23653|
|CLD      |McClellan-Palomar Airport                     |San Diego    |CA   |USA    |33.12723|-117.27873|
+---------+----------------------------------------------+-------------+-----+-------+--------+----------+
only showing top 5 rows


In [27]:
flights.filter(F.col("ORIGIN_AIRPORT") == "SFO").select(
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DISTANCE"
).show(5, truncate=False)

+--------------+-------------------+--------+
|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DISTANCE|
+--------------+-------------------+--------+
|SFO           |CLT                |2296    |
|SFO           |MSP                |1589    |
|SFO           |DFW                |1464    |
|SFO           |IAH                |1635    |
|SFO           |DEN                |967     |
+--------------+-------------------+--------+
only showing top 5 rows


In [28]:
flights_with_tsm = flights.withColumn(
    "SCHEDULED_DEPARTURE_TSM",
    F.floor(F.col("SCHEDULED_DEPARTURE") / 100) * 60 + F.col("SCHEDULED_DEPARTURE") % 100
).withColumn(
    "SCHEDULED_ARRIVAL_TSM",
    F.floor(F.col("SCHEDULED_ARRIVAL") / 100) * 60 + F.col("SCHEDULED_ARRIVAL") % 100
)

flights_with_tsm.select(
    "SCHEDULED_DEPARTURE", "SCHEDULED_DEPARTURE_TSM",
    "SCHEDULED_ARRIVAL", "SCHEDULED_ARRIVAL_TSM"
).show(5, truncate=False)

+-------------------+-----------------------+-----------------+---------------------+
|SCHEDULED_DEPARTURE|SCHEDULED_DEPARTURE_TSM|SCHEDULED_ARRIVAL|SCHEDULED_ARRIVAL_TSM|
+-------------------+-----------------------+-----------------+---------------------+
|5                  |5                      |430              |270                  |
|10                 |10                     |750              |470                  |
|20                 |20                     |806              |486                  |
|20                 |20                     |805              |485                  |
|25                 |25                     |320              |200                  |
+-------------------+-----------------------+-----------------+---------------------+
only showing top 5 rows


In [29]:
flights_with_tsm.withColumn(
    "MEAN TARGET_SPEED",
    F.col("DISTANCE") * 60 / (F.col("SCHEDULED_ARRIVAL_TSM") - F.col("SCHEDULED_DEPARTURE_TSM"))
).select("MEAN TARGET_SPEED").show(5, truncate=False)

+------------------+
|MEAN TARGET_SPEED |
+------------------+
|327.8490566037736 |
|303.9130434782609 |
|295.6223175965665 |
|302.19354838709677|
|496.45714285714286|
+------------------+
only showing top 5 rows


In [30]:
airports.filter(
    F.col("STATE").isin(["CA", "NY", "TX"]) & (F.col("LATITUDE") < 27)
).select("CITY").show(5, truncate=False)

+-----------+
|CITY       |
+-----------+
|Brownsville|
|Harlingen  |
|McAllen    |
+-----------+



Далее проведем анализ пустых полей

In [31]:
flights.show(5)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+--------+-------------------+--------------+-----------------+------------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DISTANCE|SCHEDULED_DEPARTURE|DEPARTURE_TIME|SCHEDULED_ARRIVAL|ARRIVAL_TIME|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+--------+-------------------+--------------+-----------------+------------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|2015|    1|  1|          4|     AS|           98|     N407AS|           ANC|                SEA|    1448|                  5|          2354|              430|         408|         

In [32]:
flights.groupBy("CANCELLATION_REASON").count().orderBy(F.desc("count")).show(truncate=False)

+-------------------+-------+
|CANCELLATION_REASON|count  |
+-------------------+-------+
|NULL               |5729195|
|B                  |48851  |
|A                  |25262  |
|C                  |15749  |
|D                  |22     |
+-------------------+-------+



Удалять не следует, обыкновенное поведение

In [33]:
flights.groupBy("SECURITY_DELAY").count().orderBy(F.desc("count")).show(10, truncate=False)

+--------------+-------+
|SECURITY_DELAY|count  |
+--------------+-------+
|NULL          |4755640|
|0             |1059955|
|15            |158    |
|8             |127    |
|10            |125    |
|12            |124    |
|6             |121    |
|13            |119    |
|7             |119    |
|9             |117    |
+--------------+-------+
only showing top 10 rows


Числовое поле, писать 0 ошибочно

In [34]:
flights.groupBy("WEATHER_DELAY").count().orderBy(F.desc("count")).show(10, truncate=False)

+-------------+-------+
|WEATHER_DELAY|count  |
+-------------+-------+
|NULL         |4755640|
|0            |998723 |
|6            |1649   |
|8            |1580   |
|7            |1537   |
|10           |1498   |
|15           |1498   |
|9            |1487   |
|16           |1460   |
|5            |1415   |
+-------------+-------+
only showing top 10 rows


Заменять не следует

In [37]:
flights.alias('fl').join(
    airports.alias('ap'),
    on=F.col('fl.ORIGIN_AIRPORT') == F.col('ap.IATA_CODE'),
    how='left'
).groupBy('AIRPORT', 'CITY').agg(F.count(F.lit(1)).alias('flights')).filter(F.col('CITY') == 'Los Angeles').show(5, truncate=False)

+---------------------------------+-----------+-------+
|AIRPORT                          |CITY       |flights|
+---------------------------------+-----------+-------+
|Los Angeles International Airport|Los Angeles|194673 |
+---------------------------------+-----------+-------+



In [40]:
flights.groupBy(F.col('FLIGHT_NUMBER')).agg(
    F.count(F.lit(1)).alias("count"),
    F.count_distinct('ORIGIN_AIRPORT').alias("Unique airports"),
    F.avg('DISTANCE')
).orderBy(F.desc("count")).show(5, truncate=False)

+-------------+-----+---------------+-----------------+
|FLIGHT_NUMBER|count|Unique airports|avg(DISTANCE)    |
+-------------+-----+---------------+-----------------+
|469          |3975 |53             |884.1683018867925|
|327          |3554 |48             |931.9414743950479|
|326          |3513 |48             |849.3845715912325|
|188          |3386 |39             |749.719728292971 |
|403          |3370 |38             |1172.780415430267|
+-------------+-----+---------------+-----------------+
only showing top 5 rows


In [43]:
airports.alias('ap').join(
    flights.alias('fl'),
    on=F.col('fl.ORIGIN_AIRPORT') == F.col('ap.IATA_CODE'),
    how='left'
).filter(F.isnull('FLIGHT_NUMBER')).select('AIRPORT').show(5, truncate=False)

+-------+
|AIRPORT|
+-------+
+-------+



In [48]:
airports.alias('ap').join(
    flights.alias('fl'),
    on=F.col('fl.ORIGIN_AIRPORT') == F.col('ap.IATA_CODE'),
    how='left_anti'
).select('AIRPORT').show(5, truncate=False)

+-------+
|AIRPORT|
+-------+
+-------+



In [47]:
airports.alias('ap').join(
    flights.alias('fl'),
    on=F.col('fl.ORIGIN_AIRPORT') == F.col('ap.IATA_CODE'),
    how='left'
).groupBy('AIRPORT').agg(F.count(F.lit(1)).alias('Outgoing flights')).orderBy(F.desc('Outgoing flights')).show(5, truncate=False)

+------------------------------------------------+----------------+
|AIRPORT                                         |Outgoing flights|
+------------------------------------------------+----------------+
|Hartsfield-Jackson Atlanta International Airport|346836          |
|Chicago O'Hare International Airport            |285884          |
|Dallas/Fort Worth International Airport         |239551          |
|Denver International Airport                    |196055          |
|Los Angeles International Airport               |194673          |
+------------------------------------------------+----------------+
only showing top 5 rows


In [ ]:
flights.createTempView('Flights')

AnalysisException: [TEMP_TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create the temporary view `Flights` because it already exists.
Choose a different name, drop or replace the existing view. SQLSTATE: 42P07

In [51]:
spark.sql("""\
SELECT AVG(DISTANCE)
FROM Flights;
""").show()

+-----------------+
|    avg(DISTANCE)|
+-----------------+
|822.3564947305235|
+-----------------+



In [58]:
spark.sql("""\
SELECT TAIL_NUMBER, COUNT(*) AS total_flights
FROM Flights
GROUP BY TAIL_NUMBER
HAVING TAIL_NUMBER IS NOT NULL
ORDER BY COUNT(*) DESC
""").show(5)

+-----------+-------------+
|TAIL_NUMBER|total_flights|
+-----------+-------------+
|     N480HA|         3768|
|     N488HA|         3723|
|     N484HA|         3723|
|     N493HA|         3585|
|     N478HA|         3577|
+-----------+-------------+
only showing top 5 rows


In [59]:
combined_df = flights.alias('fl').join(
    airports.alias('ap'),
    on=F.col('fl.ORIGIN_AIRPORT') == F.col('ap.IATA_CODE'),
    how='left'
)

combined_df.show(5)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+--------+-------------------+--------------+-----------------+------------+-------------------+----------------+--------------+-------------+-------------------+-------------+---------+--------------------+-------------+-----+-------+--------+----------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DISTANCE|SCHEDULED_DEPARTURE|DEPARTURE_TIME|SCHEDULED_ARRIVAL|ARRIVAL_TIME|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|IATA_CODE|             AIRPORT|         CITY|STATE|COUNTRY|LATITUDE| LONGITUDE|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+--------+-------------------+--------------+-----------------+------------+-------------------+----------------+--------------+-------------+-------------------+-------------+---------+--------------

In [61]:
out = Path('hw_output')
combined_df.write.mode('overwrite').parquet(str(out / 'combined_df'))
combined_df.count()

26/09/19 17:32:17 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


5819079

In [62]:
verify_df = spark.read.parquet(str(out / 'combined_df'))

verify_df.count()

5819079

# Выводы

1. Brownsville, Harlingen, McAllen одни из самых южных городов США, имеющих аэропорт и ведущих активные полеты
2. Всех полетов 5729195 не имели задержек, а остальные имели задержки в 4 категориях 48851 25262 15749 22. Категории AIR_SYSTEM_DELAY SECURITY_DELAY AIRLINE_DELAY LATE_AIRCRAFT_DELAY WEATHER_DELAY
3. Чаще всего вылетают из Атланты, Джорджия
4. Средняя продолжительность полета составляет 822. Единицы измерения не указаны, могут быть как мили, километры, так и морские мили.
5. N480HA - самый популярный борт, совершивший 3768 полетов

Ограничения:
1. В данных есть пропуски, в том числе на параметрах, которые не должны ошибочно читаться. Например, бортовой номер
2. Не указаны единицы измерения, они отсутствуют даже на сайте